# Horse Racing RL — Training Notebook

Train jockey AI models using curriculum learning (SB3).

**Colab tips:**
- `N_ENVS` auto-scales to available CPU cores (more cores = faster rollout collection)
- MLP policies train faster on CPU than GPU — the bottleneck is env stepping, not gradient updates
- Stages 1-8 use `SubprocVecEnv` (true multiprocessing); stages 9-11 use `DummyVecEnv` (ONNX sessions)

**Outputs:** ONNX model files in `models/v4/` for browser inference.

## 1. Setup

In [ ]:
import os

# Clone or pull latest
if os.path.exists("hr-simulation"):
    !cd hr-simulation && git pull
else:
    !git clone https://github.com/ue-too/hr-simulation.git

%cd hr-simulation

In [ ]:
# Remove old gym (Colab pre-installs it) and install dependencies
!pip uninstall -y gym 2>/dev/null
!pip install -q stable-baselines3 gymnasium pettingzoo numpy torch onnx onnxruntime tensorboard

In [ ]:
# Verify import
from horse_racing.engine import HorseRacingEngine
from horse_racing.types import HorseAction, HORSE_COUNT
from horse_racing.env import HorseRacingSingleEnv
from horse_racing.self_play_env import SelfPlayEnv
from horse_racing.reward import compute_reward
print(f"Engine OK — {HORSE_COUNT} horses")

# Quick sanity check
engine = HorseRacingEngine("tracks/test_oval.json")
engine.step([HorseAction() for _ in range(HORSE_COUNT)])
obs = engine.get_observations()
arr = engine.obs_to_array(obs[0])
print(f"Observation vector: {arr.shape} — {arr.dtype}")

## 2. Training Configuration

In [ ]:
import os

DEVICE = "cpu"
LOG_DIR = "logs/baseline"

# Auto-detect CPU cores and scale envs accordingly
cpu_count = os.cpu_count() or 2
N_ENVS = max(2, cpu_count)
# Overtake stages use DummyVecEnv (serial), so fewer envs avoids wasted serial work
N_ENVS_OVERTAKE = max(2, cpu_count // 2)
# Batch size scales with env count — keep ratio ~32 steps/env
BATCH_SIZE = max(256, N_ENVS * 32)

# Check available resources
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
    print(f"CPUs: {cpu_count} | RAM: {ram_gb:.1f} GB")
except ImportError:
    print(f"CPUs: {cpu_count}")
print(f"Device: {DEVICE} (MLP policies are faster on CPU)")
print(f"Parallel envs: {N_ENVS} (stages 1-8), {N_ENVS_OVERTAKE} (stages 9-11)")
print(f"Batch size: {BATCH_SIZE}")

# max_steps set to ~2x minimum finish ticks per track so the agent can always finish
CURRICULUM = [
    {"track": "tracks/curriculum_1_straight.json", "timesteps": 500_000, "max_steps": 1500, "name": "Stage 1: Straight"},
    {"track": "tracks/curriculum_2_gentle_oval.json", "timesteps": 750_000, "max_steps": 7000, "name": "Stage 2: Gentle oval"},
    {"track": "tracks/curriculum_3_tight_oval.json", "timesteps": 750_000, "max_steps": 4000, "name": "Stage 3: Tight oval"},
    {"track": "tracks/tokyo.json", "timesteps": 1_000_000, "max_steps": 3500, "name": "Stage 4: Tokyo"},
    {"track": "tracks/kokura.json", "timesteps": 1_000_000, "max_steps": 5500, "name": "Stage 5: Kokura"},
    {"track": "tracks/tokyo_2600.json", "timesteps": 1_000_000, "max_steps": 6000, "name": "Stage 6: Tokyo 2600"},
    {"track": "tracks/hanshin.json", "timesteps": 1_000_000, "max_steps": 4000, "name": "Stage 7: Hanshin"},
    {"track": "tracks/kyoto.json", "timesteps": 1_000_000, "max_steps": 4000, "name": "Stage 8: Kyoto"},
]

OVERTAKE_CURRICULUM = [
    {
        "tracks": ["tracks/tokyo.json"],
        "timesteps": 1_000_000,
        "max_steps": 5000,
        "min_opponents": 3,
        "max_opponents": 3,
        "stagger_range": (10.0, 30.0),
        "name": "Stage 9: Overtake (3 opponents, Tokyo)",
    },
    {
        "tracks": ["tracks/tokyo.json", "tracks/kokura.json"],
        "timesteps": 1_000_000,
        "max_steps": 5000,
        "min_opponents": 3,
        "max_opponents": 5,
        "stagger_range": (5.0, 20.0),
        "name": "Stage 10: Overtake (3-5 opponents, multi-track)",
    },
    {
        "tracks": ["tracks/tokyo.json", "tracks/kokura.json", "tracks/hanshin.json"],
        "timesteps": 1_500_000,
        "max_steps": 5000,
        "min_opponents": 3,
        "max_opponents": 7,
        "stagger_range": (0.0, 15.0),
        "name": "Stage 11: Overtake (3-7 opponents, multi-track)",
    },
]

total_all = sum(s["timesteps"] for s in CURRICULUM) + sum(s["timesteps"] for s in OVERTAKE_CURRICULUM)
num_stages = len(CURRICULUM) + len(OVERTAKE_CURRICULUM)
print(f"\nStages 1-{len(CURRICULUM)}: single-agent, Stages {len(CURRICULUM)+1}-{num_stages}: overtake")
print(f"Total timesteps: {total_all:,}")

## 3. Curriculum Training (SB3)

In [ ]:
from pathlib import Path
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
import time
import numpy as np
import torch
import torch.nn as nn


class ProgressCallback(BaseCallback):
    """Tracks reward, speed, and steering action usage during training."""
    def __init__(self, stage_name, total_timesteps, stage_idx, num_stages, print_freq=2048):
        super().__init__()
        self.stage_name = stage_name
        self.total = total_timesteps
        self.print_freq = print_freq
        self.stage_idx = stage_idx
        self.num_stages = num_stages
        self.start_time = None
        self.step_offset = 0
        self.tang_actions = []
        self.norm_actions = []

    def _on_training_start(self):
        self.start_time = time.time()
        self.step_offset = self.num_timesteps

    def _on_step(self) -> bool:
        actions = self.locals.get("actions")
        if actions is not None:
            self.tang_actions.append(float(actions[0][0]))
            self.norm_actions.append(float(actions[0][1]))

        if self.n_calls % self.print_freq == 0 and len(self.model.ep_info_buffer) > 0:
            mean_r = sum(ep['r'] for ep in self.model.ep_info_buffer) / len(self.model.ep_info_buffer)
            elapsed = time.time() - self.start_time
            stage_steps = self.num_timesteps - self.step_offset
            sps = stage_steps / elapsed if elapsed > 0 else 0
            pct = 100 * stage_steps / self.total
            overall = 100 * (self.stage_idx + stage_steps / self.total) / self.num_stages
            eta = (self.total - stage_steps) / sps if sps > 0 else 0

            recent = min(self.print_freq, len(self.tang_actions))
            tang = self.tang_actions[-recent:]
            norm = self.norm_actions[-recent:]
            norm_abs = np.mean(np.abs(norm)) if norm else 0

            print(f"  [{self.stage_name}] {pct:5.1f}% | overall: {overall:4.1f}% | steps: {stage_steps:>8,} | reward: {mean_r:8.2f} | "
                  f"tang: {np.mean(tang):+.2f} | |norm|: {norm_abs:.3f} | "
                  f"{sps:.0f} sps | ETA: {eta/60:.1f}m")
        return True


class PolicyNetwork(nn.Module):
    def __init__(self, sb3_policy):
        super().__init__()
        self.features_extractor = sb3_policy.features_extractor
        self.mlp_extractor = sb3_policy.mlp_extractor
        self.action_net = sb3_policy.action_net

    def forward(self, obs):
        features = self.features_extractor(obs)
        latent_pi, _ = self.mlp_extractor(features)
        return self.action_net(latent_pi)


def export_onnx(sb3_model, output_path: str) -> None:
    wrapper = PolicyNetwork(sb3_model.policy)
    wrapper.eval()
    obs_dim = sb3_model.observation_space.shape[0]
    dummy = torch.zeros(1, obs_dim, dtype=torch.float32)
    torch.onnx.export(
        wrapper, dummy, output_path,
        input_names=["obs"], output_names=["action"],
        dynamic_axes={"obs": {0: "batch"}, "action": {0: "batch"}},
        opset_version=17, dynamo=False,
    )


def make_env(track_path, max_steps):
    def _init():
        return Monitor(HorseRacingSingleEnv(track_path=track_path, max_steps=max_steps))
    return _init


def make_self_play_env(tracks, max_steps, opponent_onnx_paths, min_opponents, max_opponents, stagger_range):
    def _init():
        return Monitor(SelfPlayEnv(
            tracks=tracks,
            max_steps=max_steps,
            opponent_onnx_paths=opponent_onnx_paths,
            min_opponents=min_opponents,
            max_opponents=max_opponents,
            stagger_range=stagger_range,
        ))
    return _init


version = "v4"
models_dir = Path("models") / version
models_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir = Path("checkpoints/baseline")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Use SubprocVecEnv for true parallel rollouts (faster than DummyVecEnv)
VecEnvClass = SubprocVecEnv if N_ENVS > 1 else DummyVecEnv
print(f"Using {VecEnvClass.__name__} with {N_ENVS} envs\n")

model = None
total_start = time.time()

# ── Stages 1-8: Single-agent track navigation ────────────────────
for i, stage in enumerate(CURRICULUM):
    stage_num = i + 1
    print(f"{'='*60}")
    print(f"{stage['name']} — {stage['timesteps']:,} timesteps")
    print(f"{'='*60}")

    env = VecEnvClass([make_env(stage['track'], stage['max_steps']) for _ in range(N_ENVS)])

    if model is None:
        model = PPO(
            "MlpPolicy", env, verbose=0,
            n_steps=2048, batch_size=BATCH_SIZE, n_epochs=10,
            learning_rate=3e-4, gamma=0.99, gae_lambda=0.95,
            clip_range=0.2, vf_coef=0.5, ent_coef=0.01,
            device=DEVICE,
            tensorboard_log=LOG_DIR,
        )
    else:
        model.set_env(env)

    callback = ProgressCallback(stage['name'], stage['timesteps'], i, num_stages)
    model.learn(
        total_timesteps=stage['timesteps'], callback=callback,
        reset_num_timesteps=False, tb_log_name="curriculum",
    )

    save_path = checkpoint_dir / f"curriculum_stage_{stage_num}"
    model.save(str(save_path))
    onnx_path = models_dir / f"baseline_stage{stage_num}.onnx"
    export_onnx(model, str(onnx_path))
    print(f"  Saved → {save_path}")
    print(f"  ONNX  → {onnx_path}\n")
    env.close()

# ── Stages 9-11: Overtake curriculum with scripted opponents ─────
latest_onnx = str(models_dir / f"baseline_stage{len(CURRICULUM)}.onnx")

for j, stage in enumerate(OVERTAKE_CURRICULUM):
    stage_idx = len(CURRICULUM) + j
    stage_num = stage_idx + 1
    print(f"{'='*60}")
    print(f"{stage['name']} — {stage['timesteps']:,} timesteps")
    print(f"  Opponents: {latest_onnx}")
    print(f"  Stagger: {stage['stagger_range'][0]:.0f}-{stage['stagger_range'][1]:.0f}m ahead")
    print(f"  Envs: {N_ENVS_OVERTAKE} (DummyVecEnv)")
    print(f"{'='*60}")

    # SelfPlayEnv uses ONNX sessions which can't be pickled for SubprocVecEnv,
    # so use DummyVecEnv with fewer envs to limit serial overhead
    env = DummyVecEnv([
        make_self_play_env(
            tracks=stage["tracks"],
            max_steps=stage["max_steps"],
            opponent_onnx_paths=[latest_onnx],
            min_opponents=stage["min_opponents"],
            max_opponents=stage["max_opponents"],
            stagger_range=stage["stagger_range"],
        )
        for _ in range(N_ENVS_OVERTAKE)
    ])

    model.set_env(env)

    callback = ProgressCallback(stage["name"], stage["timesteps"], stage_idx, num_stages)
    model.learn(
        total_timesteps=stage["timesteps"], callback=callback,
        reset_num_timesteps=False, tb_log_name="curriculum",
    )

    save_path = checkpoint_dir / f"curriculum_stage_{stage_num}"
    model.save(str(save_path))
    onnx_path = models_dir / f"baseline_stage{stage_num}.onnx"
    export_onnx(model, str(onnx_path))
    print(f"  Saved → {save_path}")
    print(f"  ONNX  → {onnx_path}\n")
    env.close()

    latest_onnx = str(onnx_path)

# Final model = last stage
final_onnx = models_dir / "baseline.onnx"
export_onnx(model, str(final_onnx))

total_time = time.time() - total_start
print(f"Final ONNX → {final_onnx}")
print(f"Curriculum complete in {total_time/60:.1f} minutes")

## 4. Export to ONNX

In [ ]:
import onnxruntime as ort

# Verify the final ONNX model
onnx_path = str(models_dir / "baseline.onnx")
sess = ort.InferenceSession(onnx_path)
obs_dim = model.observation_space.shape[0]
test = np.zeros((1, obs_dim), dtype=np.float32)
result = sess.run(["action"], {"obs": test})
print(f"ONNX verified: {onnx_path}")
print(f"Input: (1, {obs_dim}) → Output: {result[0][0]}")

# List all exported stage models
import glob
stage_models = sorted(glob.glob(str(models_dir / "baseline_stage*.onnx")))
print(f"\nStage models ({len(stage_models)}):")
for p in stage_models:
    print(f"  {p}")

## 5. Quick Evaluation

In [ ]:
# Run a race with the trained model and check behavior
engine = HorseRacingEngine("tracks/tokyo.json")
sess = ort.InferenceSession(onnx_path)

for tick in range(3000):
    obs = engine.get_observations()
    arr = engine.obs_to_array(obs[0]).reshape(1, -1)
    action = sess.run(["action"], {"obs": arr})[0][0]
    
    actions = [HorseAction(float(action[0]), float(action[1]))]
    for _ in range(1, HORSE_COUNT):
        actions.append(HorseAction())
    engine.step(actions)
    
    if tick % 500 == 499:
        o = obs[0]
        print(
            f"Tick {tick:4d} | "
            f"progress: {o['track_progress']:.3f} | "
            f"vel: {o['tangential_vel']:.1f} | "
            f"stamina: {o['stamina_ratio']:.3f} | "
            f"disp: {o['displacement']:+.2f} | "
            f"normal_vel: {o['normal_vel']:+.3f}"
        )

print(f"\nFinal progress: {engine.horses[0].track_progress:.3f}")
print(f"Final stamina: {engine.horses[0].runtime.current_stamina:.1f}")

## 6. Download Model

Download the ONNX file to use in the browser simulation.

In [ ]:
# Download the final ONNX model and all stage models
import shutil

try:
    from google.colab import files
    # Download final model
    files.download(onnx_path)
    print(f"Downloading {onnx_path}")
    # Zip all stage models for bulk download
    shutil.make_archive("v4_stage_models", "zip", ".", str(models_dir))
    files.download("v4_stage_models.zip")
    print("Downloading v4_stage_models.zip")
except ImportError:
    print(f"Not in Colab. Models saved at: {models_dir}/")

In [ ]:
# Download TensorBoard logs for local viewing
# Run locally: tensorboard --logdir logs/baseline
import shutil
shutil.make_archive("tb_logs_baseline", "zip", ".", LOG_DIR)
try:
    from google.colab import files
    files.download("tb_logs_baseline.zip")
except ImportError:
    print(f"Not in Colab. Logs at: {LOG_DIR}/")